In [2]:
import numpy as np
import pandas as pd
from utils import precision_recall_by_bin, IV

In [3]:
import warnings

warnings.filterwarnings("ignore")

In [4]:
data = pd.read_csv('../Data/temp_training.csv')

In [5]:
def flag(series, condition):
    return condition.astype(float).where(series.notna())

def divide_positive(numerator, denominator):
    with np.errstate(divide="ignore", invalid="ignore", over="ignore"):
        ratio = numerator / denominator.where(denominator > 0)
    return ratio.replace([np.inf, -np.inf], np.nan)

def divide_all(numerator, denominator):
    with np.errstate(divide="ignore", invalid="ignore", over="ignore"):
        ratio = numerator / denominator
    return ratio.replace([np.inf, -np.inf], np.nan)

In [29]:
# 1. Size
features = data[['id','data_type','default']]
features["log_assets"] = np.log(data["asst_tot"].where(data["asst_tot"] > 0))

# 2. Profitability
features["net_profit_assets"] = divide_positive(data["profit"], data["asst_tot"])
features["operating_profit_assets"] = divide_positive(data["prof_operations"], data["asst_tot"])
features["operating_margin"] = divide_positive(data["prof_operations"], data["rev_operating"])

# 3. Liquidity
features["working_capital_assets"] = divide_positive(data["wc_net"], data["asst_tot"])
features["cash_assets"] = divide_positive(data["cash_and_equiv"], data["asst_tot"])
features["cash_short_term_debt"] = divide_positive(data["cash_and_equiv"], data["debt_st"])

# 4. Solvency and debt burden
data['liabilities_tot'] = data["asst_tot"] - data["eqty_tot"]

features["liabilities_assets_proxy"] = divide_positive(data['liabilities_tot'], data["asst_tot"])
features["equity_assets"] = divide_positive(data["eqty_tot"], data["asst_tot"])
features["short_term_debt_assets"] = divide_positive(data["debt_st"], data["asst_tot"])

# 5. Debt-service capacity
features["interest_coverage_proxy"] = divide_positive(data["prof_operations"], data["exp_financing"])

# 6. Cash generation
features["operating_cashflow_assets"] = divide_positive(data["cf_operations"], data["asst_tot"])
features["cashflow_liabilities_proxy"] = divide_positive(data["cf_operations"], data['liabilities_tot'])
features["cashflow_profit_gap_assets"] = divide_positive(data["profit"] - data["cf_operations"], data["asst_tot"])

# 7. Activity and receivables
features["sales_assets"] = divide_positive(data["rev_operating"], data["asst_tot"])
features["receivables_assets"] = divide_positive(data["AR"], data["asst_tot"])
features["receivables_days_proxy"] = 365 * divide_positive(data["AR"].where(data["AR"] >= 0), data["rev_operating"])

# 8. Distress and special-state indicators
features["negative_equity"] = flag(data["eqty_tot"], data["eqty_tot"] < 0)
features["operating_loss"] = flag(data["prof_operations"], data["prof_operations"] < 0)
features["nonpositive_revenue"] = flag(data["rev_operating"], data["rev_operating"] <= 0)
features["zero_financing_expense"] = flag(data["exp_financing"], data["exp_financing"] == 0)

# 9. Business context
features["sector_category"] = data["ateco_sector"]#.astype("string")
features["legal_structure_category"] = data["legal_struct"].astype("string")

In [31]:
feature_cols = [f for f in features.columns if f not in ['id','data_type','default']]
iv_df = []
for f in feature_cols:
    try:
        iv_train, iv_train_df = IV(features.query('data_type == "train"'), f)
        iv_test, iv_test_df = IV(features.query('data_type == "test"'), f)
        iv_oot, iv_oot_df = IV(features.query('data_type == "oot"'), f)
        iv_df.append([f, iv_train, iv_train_df, iv_test, iv_test_df, iv_oot, iv_oot_df])
    except:
        print(f'FAILED: {f}')
iv_df = pd.DataFrame(iv_df, columns=['features','iv_train','iv_train_df','iv_test','iv_test_df','iv_oot','iv_oot_df'])
iv_df.to_csv('../Files/iv_df.csv', index=False)

FAILED: legal_structure_category


In [43]:
iv_df.query('iv_train <= 0.02')

,features,iv_train,iv_train_df,iv_test,iv_test_df,iv_oot,iv_oot_df
0,log_assets,7.146279e-03,bins Goods Bads Goods (%) B...,1.272813e-02,bins Goods Bads Goods (%) B...,1.066011e-02,bins Goods Bads Goods (%) B...
17,negative_equity,2.197584e-12,bins Goods Bads Goods (%) Bad...,1.096944e-11,bins Goods Bads Goods (%) Bad...,1.501506e-07,bins Goods Bads Goods (%) Bad...
18,operating_loss,1.286662e-04,bins Goods Bads Goods (%) Bad...,1.096944e-11,bins Goods Bads Goods (%) Bad...,5.486114e-06,bins Goods Bads Goods (%) Bad...
19,nonpositive_revenue,2.630087e-05,bins Goods Bads Goods (%) Bad...,1.710884e-03,bins Goods Bads Goods (%) Bad...,1.502038e-05,bins Goods Bads Goods (%) Bad...
20,zero_financing_expense,1.641188e-07,bins Goods Bads Goods (%) Bad...,2.149781e-03,bins Goods Bads Goods (%) Bad...,2.696748e-05,bins Goods Bads Goods (%) Bad...


In [42]:
filt = iv_df.query('iv_train >= 0.5')
first_feat = filt.iloc[0]
print(f'First feature with IV >= 0.5: {first_feat["features"]} (IV={first_feat["iv_train"]})')

display(first_feat['iv_train_df'])
display(precision_recall_by_bin(features.query('data_type == "train"'), first_feat["features"]))

First feature with IV >= 0.5: net_profit_assets (IV=0.778038412590097)


,bins,Goods,Bads,Goods (%),Bads (%),WoE,IV_bin
0,"(-0.00228, 0.000842]",46940,597,0.100141,0.089776,0.109263,0.001133
1,"(-0.00999, -0.00228]",46975,562,0.100216,0.084513,0.170422,0.002676
2,"(-0.0319, -0.00999]",46828,709,0.099902,0.106618,-0.065062,0.000437
3,"(-41.421, -0.0319]",44729,2809,0.095424,0.422408,-1.487639,0.486433
4,"(0.000842, 0.0034]",46969,568,0.100203,0.085415,0.159675,0.002361
5,"(0.0034, 0.00791]",47121,416,0.100527,0.062558,0.474336,0.018010
6,"(0.00791, 0.0158]",47218,319,0.100734,0.047971,0.741879,0.039144
7,"(0.0158, 0.0306]",47328,209,0.100969,0.031430,1.167046,0.081155
8,"(0.0306, 0.0647]",47350,187,0.101016,0.028122,1.278731,0.093212
9,"(0.0647, 5.043]",47263,274,0.100830,0.041205,0.894889,0.053358


,bin,records,defaults,population (%),precision (%),recall (%),cumm_precision (%),cumm_recall (%),lift
0,"(-41.421, -0.0319]",47538,2809,10.00%,5.91%,42.24%,5.91%,42.24%,4.224
1,"(-0.0319, -0.00999]",47537,709,10.00%,1.49%,10.66%,3.70%,52.90%,2.645
2,"(-0.00999, -0.00228]",47537,562,10.00%,1.18%,8.45%,2.86%,61.35%,2.045
3,"(-0.00228, 0.000842]",47537,597,10.00%,1.26%,8.98%,2.46%,70.33%,1.758
4,"(0.000842, 0.0034]",47537,568,10.00%,1.19%,8.54%,2.21%,78.87%,1.578
5,"(0.0034, 0.00791]",47537,416,10.00%,0.88%,6.26%,1.98%,85.13%,1.419
6,"(0.00791, 0.0158]",47537,319,10.00%,0.67%,4.80%,1.80%,89.92%,1.285
7,"(0.0158, 0.0306]",47537,209,10.00%,0.44%,3.14%,1.63%,93.07%,1.163
8,"(0.0306, 0.0647]",47537,187,10.00%,0.39%,2.81%,1.49%,95.88%,1.065
9,"(0.0647, 5.043]",47537,274,10.00%,0.58%,4.12%,1.40%,100.00%,1.000
